In [1]:
import pandas as pd
import numpy as np
import os
import gc
import random
import warnings
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import KFold

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. 設定 (Configuration)
# ==============================================================================
if os.path.exists('/kaggle/input/joai-competition-2026/train.csv'):
    INPUT_DIR = '/kaggle/input/joai-competition-2026'
elif os.path.exists('/kaggle/input/ai2026/train.csv'):
    INPUT_DIR = '/kaggle/input/ai2026'
else:
    INPUT_DIR = '.' 

TRAIN_PATH = os.path.join(INPUT_DIR, 'train.csv')
TEST_PATH = os.path.join(INPUT_DIR, 'test.csv')
SUBMISSION_PATH = 'submission_rnn_vanilla.csv'

# Vanilla RNN用のハイパーパラメータ
MAX_LEN = 40
BATCH_SIZE = 64
N_FOLDS = 5
EPOCHS = 30
# RNNは学習が不安定になりやすいため、学習率は少し控えめに設定する場合もありますが、
# 今回は勾配クリッピング(clip_grad_norm)を入れるので標準的な値で攻めます。
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {DEVICE}")

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)

# ==============================================================================
# 2. 特徴量エンジニアリング (PID Features)
# ==============================================================================
def add_features(df):
    df_eng = df.copy()
    ignore_cols = ['id', 'sample_id', 'day_n', 'time', 'lever', 'mouse_id']
    base_cols = [c for c in df_eng.columns if c not in ignore_cols]
    
    grouped = df_eng.groupby('sample_id')
    
    max_day = df_eng['day_n'].max()
    gain = 1.0 + (df_eng['day_n'] / max_day * 0.5)
    
    for col in base_cols:
        # Gain
        df_eng[f'{col}_w'] = df_eng[col] * gain
        # Diff (微分)
        df_eng[f'{col}_d1'] = grouped[col].diff().fillna(0) * gain
        # Cumsum (積分) - RNNは短期記憶しか持てないため、この積分特徴量が命綱になります
        df_eng[f'{col}_cum'] = grouped[col].cumsum()
        
    return df_eng

print(">>> Loading & Preprocessing...")
train = pd.read_csv(TRAIN_PATH).fillna(0)
test = pd.read_csv(TEST_PATH).fillna(0)

train = add_features(train)
test = add_features(test)

ignore_cols = ['id', 'sample_id', 'day_n', 'time', 'lever', 'mouse_id']
train_feats = [c for c in train.columns if c not in ignore_cols]
test_feats = set(test.columns)
feature_cols = [c for c in train_feats if c in test_feats]

scaler = RobustScaler()
train[feature_cols] = scaler.fit_transform(train[feature_cols])
test[feature_cols] = scaler.transform(test[feature_cols])

# ==============================================================================
# 3. Dataset
# ==============================================================================
class BrainDataset(Dataset):
    def __init__(self, df, feature_cols, target_col=None, max_len=40):
        self.df = df
        self.sample_ids = df['sample_id'].unique()
        self.feature_cols = feature_cols
        self.target_col = target_col
        self.max_len = max_len
        self.grouped = df.groupby('sample_id')

    def __len__(self):
        return len(self.sample_ids)

    def __getitem__(self, idx):
        sample_id = self.sample_ids[idx]
        group = self.grouped.get_group(sample_id)
        x = group[self.feature_cols].values
        seq_len = len(x)
        x_pad = np.zeros((self.max_len, len(self.feature_cols)), dtype=np.float32)
        x_pad[:seq_len, :] = x
        mask = np.zeros((self.max_len,), dtype=np.float32)
        mask[:seq_len] = 1.0
        ret = {'x': torch.tensor(x_pad), 'mask': torch.tensor(mask), 'id': sample_id}
        if self.target_col:
            y = group[self.target_col].values
            y_pad = np.zeros((self.max_len,), dtype=np.float32)
            y_pad[:seq_len] = y
            ret['y'] = torch.tensor(y_pad)
        return ret

# ==============================================================================
# 4. Model Definition: Vanilla RNN
# ==============================================================================
class VanillaRNNModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, num_layers=2):
        super().__init__()
        
        # 入力層
        self.embedding = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU()
        )
        
        # 原始的なRNNレイヤー
        # nonlinearity='tanh' が標準的です ('relu'だとさらに爆発しやすい)
        self.rnn = nn.RNN(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            nonlinearity='tanh',
            batch_first=True,
            bidirectional=True,
            dropout=0.1
        )

        # 出力層
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim * 2), # Bidirectionalなので2倍
            nn.Linear(hidden_dim * 2, 128),
            nn.GELU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = self.embedding(x)
        # RNNの出力: output, h_n
        x, _ = self.rnn(x)
        out = self.head(x)
        return out.squeeze(-1)

# ==============================================================================
# 5. Training Loop
# ==============================================================================
final_test_preds = {} 
test_counts = {}

unique_samples = train['sample_id'].unique()
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
criterion = nn.MSELoss(reduction='none')

print(f"\n>>> Starting {N_FOLDS}-Fold CV with Vanilla RNN...")

test_ds = BrainDataset(test, feature_cols, None, MAX_LEN)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

for fold, (train_idx, val_idx) in enumerate(kf.split(unique_samples)):
    print(f"\n=== Fold {fold+1}/{N_FOLDS} ===")
    
    train_ids, val_ids = unique_samples[train_idx], unique_samples[val_idx]
    train_ds = BrainDataset(train[train['sample_id'].isin(train_ids)], feature_cols, 'lever', MAX_LEN)
    val_ds = BrainDataset(train[train['sample_id'].isin(val_ids)], feature_cols, 'lever', MAX_LEN)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    # モデル初期化
    model = VanillaRNNModel(
        input_dim=len(feature_cols),
        hidden_dim=256,
        num_layers=2 # 深すぎると勾配消失するので2層程度推奨
    ).to(DEVICE)
    
    if torch.cuda.device_count() > 1: model = nn.DataParallel(model)
        
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    
    best_loss = float('inf')
    best_model_path = f"best_rnn_vanilla_fold{fold}.pth"
    
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for batch in train_loader:
            x, y, m = batch['x'].to(DEVICE), batch['y'].to(DEVICE), batch['mask'].to(DEVICE)
            optimizer.zero_grad()
            preds = model(x)
            loss = (criterion(preds, y) * m).sum() / m.sum()
            loss.backward()
            
            # ★重要★ RNNは勾配爆発しやすいため、クリッピングを強め(1.0以下)にかけます
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            train_loss += loss.item()
        scheduler.step()
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                x, y, m = batch['x'].to(DEVICE), batch['y'].to(DEVICE), batch['mask'].to(DEVICE)
                preds = model(x)
                loss = (criterion(preds, y) * m).sum() / m.sum()
                val_loss += loss.item()
        
        avg_val = val_loss / len(val_loader)
        if avg_val < best_loss:
            best_loss = avg_val
            torch.save(model.state_dict(), best_model_path)
            
        if (epoch+1) % 5 == 0 or epoch == 0:
            print(f"  Ep {epoch+1:2d} | Train: {train_loss/len(train_loader):.4f} | Val: {avg_val:.4f}")

    print(f"  >> Best Val: {best_loss:.5f}")

    # Inference & Intermediate Save
    try:
        model.load_state_dict(torch.load(best_model_path))
        model.eval()
        with torch.no_grad():
            for batch in test_loader:
                x = batch['x'].to(DEVICE)
                mask = batch['mask'].to(DEVICE)
                s_ids = batch['id']
                preds = model(x).cpu().numpy()
                mask = mask.cpu().numpy()
                for i, s_id in enumerate(s_ids):
                    v_len = int(mask[i].sum())
                    for t, val in enumerate(preds[i, :v_len]):
                        k = f"{s_id}_{t}"
                        final_test_preds[k] = final_test_preds.get(k, 0) + val
                        test_counts[k] = test_counts.get(k, 0) + 1
        
        # 中間結果保存
        temp_results = [{'id': k, 'lever': max(0, v / test_counts[k])} for k, v in final_test_preds.items()]
        pd.DataFrame(temp_results).to_csv(f'temp_submission_rnn_fold{fold+1}.csv', index=False)
        print(f"  -> Saved temp_submission_rnn_fold{fold+1}.csv")
        
    except Exception as e:
        print(f"  Error in Inference Fold {fold+1}: {e}")

# ==============================================================================
# 6. Final Submission
# ==============================================================================
if len(final_test_preds) > 0:
    results = [{'id': k, 'lever': max(0, v / test_counts[k])} for k, v in final_test_preds.items()]
    df_sub = pd.DataFrame(results)
    df_sub.to_csv(SUBMISSION_PATH, index=False)
    print(f"\nSUCCESS: {SUBMISSION_PATH} created.")
else:
    print("ERROR: No predictions generated.")

Using Device: cuda
>>> Loading & Preprocessing...

>>> Starting 5-Fold CV with Vanilla RNN...

=== Fold 1/5 ===
  Ep  1 | Train: 1.3161 | Val: 1.2165
  Ep  5 | Train: 0.9211 | Val: 0.9969
  Ep 10 | Train: 0.8085 | Val: 1.0469
  Ep 15 | Train: 0.7153 | Val: 0.9207
  Ep 20 | Train: 0.6098 | Val: 0.9157
  Ep 25 | Train: 0.4908 | Val: 0.9787
  Ep 30 | Train: 0.4390 | Val: 0.9979
  >> Best Val: 0.91549
  -> Saved temp_submission_rnn_fold1.csv

=== Fold 2/5 ===
  Ep  1 | Train: 1.3209 | Val: 1.1412
  Ep  5 | Train: 0.9572 | Val: 0.9537
  Ep 10 | Train: 0.8686 | Val: 0.8640
  Ep 15 | Train: 0.7275 | Val: 0.9265
  Ep 20 | Train: 0.6111 | Val: 0.8157
  Ep 25 | Train: 0.4898 | Val: 0.8483
  Ep 30 | Train: 0.4344 | Val: 0.8561
  >> Best Val: 0.79974
  -> Saved temp_submission_rnn_fold2.csv

=== Fold 3/5 ===
  Ep  1 | Train: 1.3118 | Val: 1.0976
  Ep  5 | Train: 0.9628 | Val: 1.0965
  Ep 10 | Train: 0.8413 | Val: 0.9315
  Ep 15 | Train: 0.7329 | Val: 0.9252
  Ep 20 | Train: 0.6372 | Val: 0.9161
  